<a href="https://colab.research.google.com/github/yasuhisasugiura-crypto/PARC2026_pre/blob/main/add.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# 修正版セル2〜4（統合版）：全コピー + typing.Selfパッチ
# ============================================
import shutil
import re
from pathlib import Path

# 元のlerobot
SRC_LEROBOT = Path("/content/lerobot/src/lerobot")

# 提出フォルダをクリーンに準備
SUBMISSION_DIR = Path("/content/submission")
if SUBMISSION_DIR.exists():
    shutil.rmtree(SUBMISSION_DIR)
SUBMISSION_DIR.mkdir(parents=True)

# 抽出先
DST_LEROBOT = SUBMISSION_DIR / "lerobot"

# ⭐ 変更点：lerobot全部をコピー（ただし__pycache__は除外）
def ignore_patterns(dir, files):
    return [f for f in files if f == "__pycache__" or f.endswith(".pyc")]

shutil.copytree(SRC_LEROBOT, DST_LEROBOT, ignore=ignore_patterns)

# サイズ確認
total_size = sum(f.stat().st_size for f in DST_LEROBOT.rglob("*") if f.is_file())
print(f"📊 lerobot 全コピー後サイズ: {total_size / 1024 / 1024:.1f} MB")

# ⭐ typing.Self を typing_extensions.Self にパッチ
def patch_typing_self(directory):
    """
    ディレクトリ内の全.pyファイルで typing.Self を typing_extensions.Self に置換
    """
    patched_files = []

    for py_file in Path(directory).rglob("*.py"):
        content = py_file.read_text(encoding="utf-8")
        original = content

        # パターン: from typing import ..., Self, ...
        pattern = r"from typing import ([^;\n]*)"

        def replace_typing_self(m):
            imports = m.group(1)
            # Selfが含まれていなければ変更しない
            if "Self" not in imports.split():
                # ", Self" や "Self," も検出
                if not re.search(r"\bSelf\b", imports):
                    return m.group(0)

            # Selfを除去
            parts = [p.strip() for p in imports.split(",")]
            parts_without_self = [p for p in parts if p != "Self" and p.strip() != "Self"]

            if parts_without_self:
                new_typing_import = f"from typing import {', '.join(parts_without_self)}"
                return f"{new_typing_import}\nfrom typing_extensions import Self"
            else:
                return "from typing_extensions import Self"

        content = re.sub(pattern, replace_typing_self, content)

        if content != original:
            py_file.write_text(content, encoding="utf-8")
            patched_files.append(py_file.name)

    return patched_files

patched_files = patch_typing_self(DST_LEROBOT)
print(f"\n✅ typing.Self をパッチしたファイル: {len(patched_files)}件")

# 確認：まだ残っているか？
import subprocess
result = subprocess.run(
    ["grep", "-R", "-l", "from typing import.*Self", str(DST_LEROBOT)],
    capture_output=True, text=True,
)
remaining = result.stdout.strip().splitlines() if result.stdout.strip() else []
if remaining:
    print(f"\n⚠️ まだ残っているファイル ({len(remaining)}件):")
    for f in remaining[:5]:
        print(f"   {f}")
else:
    print("\n✅ typing.Self の直接import はすべて修正済み")

# ⭐ __version__.py の存在確認
version_file = DST_LEROBOT / "__version__.py"
if version_file.exists():
    print(f"\n✅ __version__.py 存在: {version_file}")
    print(f"   内容: {version_file.read_text()[:200]}")
else:
    print(f"\n⚠️ __version__.py が存在しません")
    # __version__.py を自作
    version_file.write_text('__version__ = "0.6.0"\n', encoding="utf-8")
    print(f"   → 自作しました")